# TFM Deteccion - Cross-generator sobre GenImage (E2)

Evaluacion de los 3 detectores entrenados sobre ProGAN (ResNet-50, ViT-B/16, UFD) frente a imagenes generadas por modelos de **difusion**. No se reentrena: solo inferencia sobre la particion `val/` de las arquitecturas de difusion de GenImage.

Documentacion detallada del experimento: `expEjecucionGenImage.md`.

Tiempo total esperado en Colab Free T4: ~3-4 h.

## 1. Setup - Drive, repo, dependencias, 7z

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!rm -rf /content/tfm_deteccion
!git clone https://github.com/manuelpalasanchez/tfm_deteccion.git
%cd /content/tfm_deteccion
!git pull

In [ ]:
!pip install -q -r requirements.txt
# p7zip-full es necesario para descomprimir los ZIP multi-volumen de GenImage
!apt-get install -y -q p7zip-full

## 2. Autenticacion Drive API y localizacion de la raiz de GenImage

Pega el folder_id de la carpeta de Drive que contiene las 8 subcarpetas de GenImage (`Midjourney`, `Stable Diffusion V1.4`, etc.). Lo sacas del enlace: `https://drive.google.com/drive/folders/<FOLDER_ID>`.

In [ ]:
from google.colab import auth as colab_auth
colab_auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build

creds, _ = google.auth.default()
service = build('drive', 'v3', credentials=creds)

GENIMAGE_ROOT_FOLDER_ID = '1jGt10bwTbhEZuGXLyvrCuxOI0cBqQ1FS'

# Verificar que el folder_id es accesible y resolver shortcut si lo es
meta = service.files().get(
    fileId=GENIMAGE_ROOT_FOLDER_ID,
    fields='id,name,mimeType,shortcutDetails',
    supportsAllDrives=True,
).execute()

if meta.get('mimeType') == 'application/vnd.google-apps.shortcut':
    target_id = meta['shortcutDetails']['targetId']
    print(f"El folder es un shortcut; resolviendo a target_id={target_id}")
    GENIMAGE_ROOT_FOLDER_ID = target_id
    meta = service.files().get(
        fileId=GENIMAGE_ROOT_FOLDER_ID,
        fields='id,name,mimeType',
        supportsAllDrives=True,
    ).execute()

print(f"Carpeta raiz GenImage: {meta['name']} (id={meta['id']})")

## 3. Listado de arquitecturas y empaquetado de volumenes

Lista las subcarpetas (una por arq) y, dentro de cada una, los ficheros `.zNN` + `.zip` que forman el ZIP multi-volumen. BigGAN se excluye (ya esta cubierta en E1b).

In [ ]:
import re

# Arqs excluidas por decision metodologica (no por capacidad de computo):
#   - biggan: ya cubierta en E1b (CNN_synth_testset)
#   - midjourney: resolucion nativa 1024 px, el resize a 224 del pipeline destruye su
#     senal caracteristica (ver expEjecucionGenImage.md seccion 8.1). Para incluirla
#     habria que adaptar el pipeline a su resolucion nativa; quitar 'midjourney' del set.
EXCLUIR_ARQS = {'biggan', 'midjourney'}

def list_children(folder_id):
    out = []
    page_token = None
    while True:
        resp = service.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields='nextPageToken, files(id,name,mimeType,size,shortcutDetails)',
            pageSize=200,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        out.extend(resp.get('files', []))
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
    return out

def resolve_shortcut(f):
    if f.get('mimeType') == 'application/vnd.google-apps.shortcut':
        target_id = f['shortcutDetails']['targetId']
        return service.files().get(
            fileId=target_id,
            fields='id,name,mimeType,size',
            supportsAllDrives=True,
        ).execute()
    return f

subdirs = [c for c in list_children(GENIMAGE_ROOT_FOLDER_ID)
           if c['mimeType'] == 'application/vnd.google-apps.folder' or c.get('mimeType') == 'application/vnd.google-apps.shortcut']

arq_info = {}
for sd in subdirs:
    sd_resolved = resolve_shortcut(sd)
    nombre = sd['name']
    nombre_key = re.sub(r'[^a-z0-9]+', '_', nombre.lower()).strip('_')
    if any(ex in nombre_key for ex in EXCLUIR_ARQS):
        print(f"  [skip] {nombre} (excluida)")
        continue

    volumes = []
    for f in list_children(sd_resolved['id']):
        f_resolved = resolve_shortcut(f)
        ext = f_resolved['name'].lower().rsplit('.', 1)[-1] if '.' in f_resolved['name'] else ''
        if ext == 'zip' or re.match(r'z\d+$', ext):
            volumes.append({
                'name': f_resolved['name'],
                'id':   f_resolved['id'],
                'size': int(f_resolved.get('size', 0)),
                'ext':  ext,
            })

    if not volumes:
        print(f"  [warn] {nombre} sin volumenes .zNN/.zip")
        continue

    # Ordenar: z01, z02, ..., zNN, zip al final
    def vol_sort_key(v):
        if v['ext'] == 'zip':
            return (1, 0)
        return (0, int(v['ext'][1:]))
    volumes.sort(key=vol_sort_key)

    total = sum(v['size'] for v in volumes)
    arq_info[nombre_key] = {
        'nombre_drive': nombre,
        'volumes':      volumes,
        'total_bytes':  total,
    }

# Imprimir tabla
print(f"\n{'Arquitectura':<35s} {'Vols':>5s}  {'GB':>8s}")
print('-' * 55)
for k, info in sorted(arq_info.items(), key=lambda kv: kv[1]['total_bytes']):
    gb = info['total_bytes'] / 1024**3
    print(f"{info['nombre_drive']:<35s} {len(info['volumes']):>5d}  {gb:>8.2f}")
print(f"\nTotal arquitecturas a evaluar: {len(arq_info)}")

## 4. Descarga y extraccion del val por arquitectura

Procesa una arq cada vez: descarga volumenes, descomprime solo `val/` con 7z y borra los volumenes. Si una arq excede el espacio libre menos un margen de seguridad, se salta con aviso (tipico para Midjourney).

Tras este bucle, `/content/genimage/{arch}/val/{ai,nature}/` contiene las imagenes de las arqs que pasaron el check.

In [ ]:
import os, shutil, subprocess, time
from googleapiclient.http import MediaIoBaseDownload

GENIMAGE_BASE = '/content/genimage'
ZIPS_DIR      = '/content/genimage_zips'
MARGEN_GB     = 15  # margen de seguridad sobre el disco libre

os.makedirs(GENIMAGE_BASE, exist_ok=True)

def disco_libre_gb(path='/content'):
    stat = shutil.disk_usage(path)
    return stat.free / 1024**3

def descargar_fichero(file_id, dest_path):
    request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with open(dest_path, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request, chunksize=64 * 1024 * 1024)
        done = False
        last_pct = -1
        while not done:
            status, done = downloader.next_chunk()
            if status:
                pct = int(status.progress() * 100)
                if pct >= last_pct + 5:
                    print(f"    {pct}%", end=' ', flush=True)
                    last_pct = pct
    print()

def procesar_arq(arq_key, info):
    nombre = info['nombre_drive']
    gb_arq = info['total_bytes'] / 1024**3
    libre  = disco_libre_gb()
    print(f"\n=== {nombre} ({gb_arq:.1f} GB, libres en disco: {libre:.1f} GB) ===")

    if gb_arq + MARGEN_GB > libre:
        print(f"  [SKIP] {gb_arq:.1f} GB + margen {MARGEN_GB} GB > libres {libre:.1f} GB")
        return False

    # Limpiar directorio de zips por si quedo algo
    if os.path.isdir(ZIPS_DIR):
        shutil.rmtree(ZIPS_DIR)
    os.makedirs(ZIPS_DIR, exist_ok=True)

    # Descargar volumenes
    t0 = time.time()
    for vol in info['volumes']:
        dest = os.path.join(ZIPS_DIR, vol['name'])
        size_gb = vol['size'] / 1024**3
        print(f"  -> {vol['name']} ({size_gb:.2f} GB)")
        descargar_fichero(vol['id'], dest)
    t_dl = time.time() - t0
    print(f"  Descarga completa en {t_dl/60:.1f} min")

    # Localizar el .zip final
    zip_final = [v['name'] for v in info['volumes'] if v['ext'] == 'zip']
    if not zip_final:
        print("  [ERROR] No hay volumen .zip final con central directory")
        shutil.rmtree(ZIPS_DIR)
        return False
    zip_path = os.path.join(ZIPS_DIR, zip_final[0])

    # Extraer solo val/ con 7z
    print(f"  Extrayendo solo val/ ...")
    t0 = time.time()
    cmd = ['7z', 'x', '-y', f'-o{GENIMAGE_BASE}', zip_path, '*/val/*']
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        # 7z code 1 = warnings (p.ej. central directory roto); code 2+ = errores
        if res.returncode > 1:
            print(f"  [ERROR] 7z fallo (code {res.returncode}):")
            print(res.stderr[-2000:])
            shutil.rmtree(ZIPS_DIR)
            return False
        print(f"  [warn] 7z code 1 (avisos), continuando")
    t_ex = time.time() - t0
    print(f"  Extraccion en {t_ex/60:.1f} min")

    # Borrar zips para liberar espacio
    shutil.rmtree(ZIPS_DIR)
    print(f"  Zips borrados. Disco libre tras la arq: {disco_libre_gb():.1f} GB")
    return True

# Bucle: procesar de menor a mayor (para llenar tier 1 antes de intentar tier 2/3)
resultados_extraccion = {}
for arq_key, info in sorted(arq_info.items(), key=lambda kv: kv[1]['total_bytes']):
    ok = procesar_arq(arq_key, info)
    resultados_extraccion[arq_key] = ok

print("\n=== Resumen extraccion ===")
for k, ok in resultados_extraccion.items():
    print(f"  {'OK ' if ok else 'SKIP'}  {k}")

In [ ]:
# Sanity check del root /content/genimage tras la extraccion
import os

if os.path.isdir(GENIMAGE_BASE):
    print(f"Contenido de {GENIMAGE_BASE}:")
    for arch_dir in sorted(os.listdir(GENIMAGE_BASE)):
        full = os.path.join(GENIMAGE_BASE, arch_dir)
        if not os.path.isdir(full):
            continue
        val_dir = os.path.join(full, 'val')
        if not os.path.isdir(val_dir):
            print(f"  {arch_dir}: no tiene val/")
            continue
        ai_dir     = os.path.join(val_dir, 'ai')
        nature_dir = os.path.join(val_dir, 'nature')
        n_ai     = sum(1 for _ in os.scandir(ai_dir))     if os.path.isdir(ai_dir)     else 0
        n_nature = sum(1 for _ in os.scandir(nature_dir)) if os.path.isdir(nature_dir) else 0
        print(f"  {arch_dir:<35s}  ai={n_ai:>6d}  nature={n_nature:>6d}")
else:
    print(f"AVISO: {GENIMAGE_BASE} no existe. Ninguna arquitectura paso el check de espacio.")

## 5. Carga de los 3 modelos desde Drive

In [ ]:
import sys, glob, importlib
from pathlib import Path

sys.path.insert(0, '/content/tfm_deteccion')

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from utils.config import load_config
from models import model_registry
import data.genimage_dataset  # registra GenImageDataset
from data.genimage_dataset import GenImageDataset
from data.transforms import get_eval_transforms
from evaluation.metrics import compute_metrics

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT_BASE = '/content/drive/MyDrive/tfm-checkpoints'
OUT_DIR   = Path(CKPT_BASE) / 'figuras_memoria_genimage'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Salidas iran a: {OUT_DIR}")

def localizar_checkpoint(prefix):
    cands = sorted(glob.glob(f'{CKPT_BASE}/{prefix}_*/checkpoint_best.pth'))
    if not cands:
        raise FileNotFoundError(f"No hay checkpoint para prefix='{prefix}'")
    return cands[-1]

modelos_info = {
    'ResNet-50': {
        'config':     Path('/content/tfm_deteccion/configs/resnet50.yaml'),
        'module':     'models.resnet50',
        'csv_key':    'resnet50',
        'color':      '#1976D2',
        'checkpoint': localizar_checkpoint('resnet50'),
    },
    'ViT-B/16': {
        'config':     Path('/content/tfm_deteccion/configs/vit.yaml'),
        'module':     'models.vit',
        'csv_key':    'vit',
        'color':      '#388E3C',
        'checkpoint': localizar_checkpoint('vit'),
    },
    'UniversalFakeDetect': {
        'config':     Path('/content/tfm_deteccion/configs/universalfakedetect.yaml'),
        'module':     'models.universalfakedetect',
        'csv_key':    'ufd',
        'color':      '#F57C00',
        'checkpoint': localizar_checkpoint('universalfakedetect'),
    },
}

for n, i in modelos_info.items():
    print(f"  {n:<22s}  checkpoint -> {i['checkpoint']}")

## 6. Inferencia

Para cada modelo: instancia, carga checkpoint, hace forward pass batch a batch sobre todo el GenImage val/. Guarda predicciones en `predicciones_{modelo}.pt` (en Drive) para permitir regenerar figuras sin reinferir.

In [ ]:
import numpy as np

predicciones = {}  # nombre -> dict con y_true, y_score, generators (arrays/listas)

for nombre, info in modelos_info.items():
    print(f"\n{'=' * 55}\n  Modelo: {nombre}\n{'=' * 55}")

    cfg = load_config(info['config'])
    importlib.import_module(info['module'])

    model_kwargs = vars(cfg.model.kwargs) if hasattr(cfg.model, 'kwargs') else {}
    model = model_registry.build(cfg.model.name, **model_kwargs)
    ckpt = torch.load(str(info['checkpoint']), map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(DEVICE).eval()
    best_val_auc = ckpt.get('best_auc', float('nan'))
    print(f"  Checkpoint cargado (val best_auc={best_val_auc:.4f})")

    dataset = GenImageDataset(
        root=GENIMAGE_BASE,
        split='val',
        transform=get_eval_transforms(),
    )
    print(f"  Dataset GenImage val: {len(dataset)} imagenes")
    print(f"  Generadores presentes: {dataset.generators}")

    loader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    all_scores, all_targets, all_gens = [], [], []
    with torch.no_grad():
        for images, labels, gens in tqdm(loader, desc=f'  {nombre}'):
            images = model.preprocess(images.to(DEVICE))
            logits = model(images).squeeze(1)
            scores = torch.sigmoid(logits).cpu().numpy()
            all_scores.extend(scores.tolist())
            all_targets.extend(labels.numpy().tolist())
            all_gens.extend(list(gens))

    y_true  = np.array(all_targets, dtype=np.int64)
    y_score = np.array(all_scores,  dtype=np.float32)
    print(f"  Inferencia completa: {len(y_true)} samples ({int(y_true.sum())} fake, {int((1 - y_true).sum())} real)")

    predicciones[nombre] = {
        'y_true':     y_true,
        'y_score':    y_score,
        'generators': all_gens,
    }

    # Guardar predicciones en Drive para regenerar figuras sin reinferir
    out_pt = OUT_DIR / f"predicciones_{info['csv_key']}.pt"
    torch.save({
        'y_true':     torch.from_numpy(y_true),
        'y_score':    torch.from_numpy(y_score),
        'generators': all_gens,
    }, str(out_pt))
    print(f"  Predicciones guardadas en {out_pt}")

    # Liberar GPU
    del model
    torch.cuda.empty_cache()

## 7. Metricas agregadas + desglose por arquitectura

Calcula AUC/AP/Accuracy/matriz de confusion para cada modelo (a) sobre todo GenImage val y (b) por arquitectura por separado. Guarda en JSON y CSV en Drive.

In [ ]:
import json, csv
from collections import defaultdict

# Mapping de nombres bonitos para figuras
NOMBRES_BONITOS = {
    'adm':                    'ADM',
    'ADM':                    'ADM',
    'glide':                  'GLIDE',
    'GLIDE':                  'GLIDE',
    'vqdm':                   'VQDM',
    'VQDM':                   'VQDM',
    'wukong':                 'Wukong',
    'Wukong':                 'Wukong',
    'midjourney':             'Midjourney',
    'Midjourney':             'Midjourney',
    'stable_diffusion_v_1_4': 'SD v1.4',
    'stable_diffusion_v1_4':  'SD v1.4',
    'stable_diffusion_v_1_5': 'SD v1.5',
    'stable_diffusion_v1_5':  'SD v1.5',
}
def nombre_bonito(gen):
    key = gen.lower().replace(' ', '_').replace('.', '_')
    return NOMBRES_BONITOS.get(gen) or NOMBRES_BONITOS.get(key) or gen

metricas_global = {}
metricas_por_arq = {}  # nombre_modelo -> {arq -> dict_metricas}

for nombre, preds in predicciones.items():
    y_true     = preds['y_true']
    y_score    = preds['y_score']
    generators = preds['generators']

    # Agregado
    metricas_global[nombre] = compute_metrics(y_true, y_score)

    # Por arquitectura: solo tiene sentido si hay tanto real como fake en la arq.
    # En GenImage, cada arq tiene su propio split val/{ai,nature}, asi que esto siempre se cumple.
    arqs_unicas = sorted(set(generators))
    metricas_por_arq[nombre] = {}
    for arq in arqs_unicas:
        mask = np.array([g == arq for g in generators])
        if mask.sum() == 0:
            continue
        yt = y_true[mask]
        ys = y_score[mask]
        if len(set(yt)) < 2:
            # Si solo hay reales o solo fakes en esta arq, no podemos calcular AUC
            metricas_por_arq[nombre][arq] = {'n_samples': int(mask.sum()), 'note': 'single class'}
            continue
        metricas_por_arq[nombre][arq] = compute_metrics(yt, ys)

# Guardar JSON
json_out = {
    'agregado': metricas_global,
    'por_arquitectura': metricas_por_arq,
}
json_path = OUT_DIR / 'metricas_genimage.json'
with json_path.open('w') as f:
    json.dump(json_out, f, indent=2)
print(f"JSON guardado: {json_path}")

# Guardar CSV (una fila por arq, columnas: n + 3 metricas x 3 modelos)
all_arqs = sorted(set().union(*[set(d.keys()) for d in metricas_por_arq.values()]))
csv_cols = ['arquitectura_gan', 'n_samples',
            'resnet50_auc', 'resnet50_ap', 'resnet50_acc',
            'vit_auc',      'vit_ap',      'vit_acc',
            'ufd_auc',      'ufd_ap',      'ufd_acc']
csv_key_map = {'ResNet-50': 'resnet50', 'ViT-B/16': 'vit', 'UniversalFakeDetect': 'ufd'}
csv_path = OUT_DIR / 'desglose_genimage_por_arquitectura.csv'
with csv_path.open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=csv_cols, extrasaction='ignore')
    w.writeheader()
    for arq in all_arqs:
        row = {'arquitectura_gan': nombre_bonito(arq)}
        n = None
        for modelo, key in csv_key_map.items():
            m = metricas_por_arq[modelo].get(arq, {})
            if 'auc_roc' in m:
                row[f'{key}_auc'] = round(m['auc_roc'], 4)
                row[f'{key}_ap']  = round(m['average_precision'], 4)
                row[f'{key}_acc'] = round(m['accuracy'], 4)
                n = m.get('n_samples', n)
        row['n_samples'] = n if n is not None else 0
        w.writerow(row)
print(f"CSV guardado: {csv_path}\n")

# Print tabla en consola
print(f"{'Arquitectura':<14s}  {'N':>6s}  {'ResNet AUC':>11s}  {'ViT AUC':>8s}  {'UFD AUC':>8s}")
print('-' * 60)
for arq in all_arqs:
    bonito = nombre_bonito(arq)
    mR = metricas_por_arq['ResNet-50'].get(arq, {}).get('auc_roc', float('nan'))
    mV = metricas_por_arq['ViT-B/16'].get(arq, {}).get('auc_roc', float('nan'))
    mU = metricas_por_arq['UniversalFakeDetect'].get(arq, {}).get('auc_roc', float('nan'))
    n  = metricas_por_arq['ResNet-50'].get(arq, {}).get('n_samples', 0)
    print(f"{bonito:<14s}  {n:>6d}  {mR:>11.4f}  {mV:>8.4f}  {mU:>8.4f}")

print("\nAgregado E2 (todas las arqs):")
for n, m in metricas_global.items():
    print(f"  {n:<22s}  AUC={m['auc_roc']:.4f}  AP={m['average_precision']:.4f}  Acc={m['accuracy']:.4f}  N={m['n_samples']}")

## 8. Figuras

Misma bateria que para E1b adaptada a E2, mas una figura adicional comparativa GAN vs difusion. Salidas en `OUT_DIR` a 300 dpi.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve

plt.rcParams.update({'font.family': 'serif', 'axes.spines.top': False, 'axes.spines.right': False})

MODELOS  = list(modelos_info.keys())
COLORES  = {n: i['color'] for n, i in modelos_info.items()}

In [ ]:
# Figura 1 - Matrices de confusion E2 comparativas
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
tick_labels = ['Real', 'Fake']

for ax, modelo in zip(axes, MODELOS):
    cm = np.array(metricas_global[modelo]['confusion_matrix'])
    cm_norm = cm.astype(float) / cm.max()
    sns.heatmap(cm, annot=False, cmap='Blues',
                xticklabels=tick_labels, yticklabels=tick_labels,
                ax=ax, cbar=False, linewidths=0.5, linecolor='white')
    for i in range(2):
        for j in range(2):
            val = cm[i, j]
            val_str = f'{val:,}'.replace(',', '.')
            text_color = 'white' if cm_norm[i, j] > 0.5 else 'black'
            ax.text(j + 0.5, i + 0.5, val_str,
                    ha='center', va='center', fontsize=13, fontweight='bold', color=text_color)
    ax.set_title(modelo, fontsize=13, pad=10)
    ax.set_xlabel('Predicho', fontsize=11)
    ax.set_ylabel('Clase real', fontsize=11)

n_total = metricas_global[MODELOS[0]]['n_samples']
fig.suptitle(f'Matrices de confusion E2 (GenImage val, N={n_total:,})', fontsize=15, y=1.00)
plt.tight_layout()
out_path = OUT_DIR / 'matrices_confusion_e2.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {out_path}")

In [ ]:
# Figura 2 - Barras de metricas E2
fig, ax = plt.subplots(figsize=(10, 6))

metricas_keys   = ['auc_roc', 'average_precision', 'accuracy']
metricas_labels = ['AUC-ROC', 'AP',                 'Accuracy']
bar_colors      = ['#1976D2', '#388E3C',            '#F57C00']
width = 0.25
x = np.arange(len(MODELOS))
offsets = [-width, 0.0, width]

for key, label, color, offset in zip(metricas_keys, metricas_labels, bar_colors, offsets):
    valores = [metricas_global[m][key] for m in MODELOS]
    bars = ax.bar(x + offset, valores, width, label=label, color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, valores):
        fmt_str = f'{val * 100:.2f}%' if key == 'accuracy' else f'{val:.4f}'
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                fmt_str, ha='center', va='bottom', fontsize=8.5)

ax.set_ylim(0.0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(MODELOS, fontsize=11)
ax.set_ylabel('Valor de la metrica', fontsize=12)
ax.set_title('Comparativa de metricas - Evaluacion E2 (cross-generator difusion)', fontsize=13)
ax.legend(fontsize=11, loc='lower right')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
out_path = OUT_DIR / 'barras_metricas_e2.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {out_path}")

In [ ]:
# Figura 3 - Curvas ROC E2 superpuestas
fig, ax = plt.subplots(figsize=(8, 8))
for nombre in MODELOS:
    preds = predicciones[nombre]
    fpr, tpr, _ = roc_curve(preds['y_true'], preds['y_score'])
    auc = metricas_global[nombre]['auc_roc']
    ax.plot(fpr, tpr, color=COLORES[nombre], lw=2.5,
            label=f'{nombre} (AUC = {auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Clasificador aleatorio')
ax.set_xlabel('Tasa de falsos positivos', fontsize=13)
ax.set_ylabel('Tasa de verdaderos positivos', fontsize=13)
ax.set_title('Curvas ROC - Evaluacion E2 (cross-generator difusion)', fontsize=14)
ax.legend(fontsize=11, loc='lower right')
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
ax.set_aspect('equal')
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
out_path = OUT_DIR / 'roc_curves_e2_superpuestas.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {out_path}")

In [ ]:
# Figura 4 - Heatmap AUC por arq generativa x modelo detector
csv_key_map_local = {'ResNet-50': 'resnet50', 'ViT-B/16': 'vit', 'UniversalFakeDetect': 'ufd'}
arqs_validas = [a for a in all_arqs if metricas_por_arq['ResNet-50'].get(a, {}).get('auc_roc') is not None]

# Ordenar por AUC media descendente
def auc_media(arq):
    valores = [metricas_por_arq[m].get(arq, {}).get('auc_roc', np.nan) for m in MODELOS]
    valores = [v for v in valores if not np.isnan(v)]
    return np.mean(valores) if valores else 0.0
arqs_sorted = sorted(arqs_validas, key=auc_media, reverse=True)

heat = np.array([
    [metricas_por_arq[m].get(a, {}).get('auc_roc', np.nan) for m in MODELOS]
    for a in arqs_sorted
])

fig, ax = plt.subplots(figsize=(8, max(4, len(arqs_sorted) * 0.8)))
sns.heatmap(
    heat, annot=False, cmap='RdYlGn', vmin=0.5, vmax=1.0,
    xticklabels=MODELOS, yticklabels=[nombre_bonito(a) for a in arqs_sorted],
    ax=ax, linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'AUC-ROC', 'shrink': 0.6},
)
for i, arq in enumerate(arqs_sorted):
    for j, modelo in enumerate(MODELOS):
        val = metricas_por_arq[modelo].get(arq, {}).get('auc_roc', np.nan)
        if not np.isnan(val):
            norm = (val - 0.5) / 0.5
            text_color = 'white' if norm < 0.15 or norm > 0.85 else 'black'
            ax.text(j + 0.5, i + 0.5, f'{val:.4f}',
                    ha='center', va='center', fontsize=9, fontweight='bold', color=text_color)

ax.set_title('AUC-ROC por arquitectura de difusion y modelo detector', fontsize=13, pad=12)
ax.set_xlabel('Modelo detector', fontsize=12)
ax.set_ylabel('Arquitectura generativa (ordenada por AUC media)', fontsize=12)
ax.tick_params(axis='x', rotation=15, labelsize=10)
ax.tick_params(axis='y', rotation=0,  labelsize=10)
plt.tight_layout()
out_path = OUT_DIR / 'heatmap_auc_por_generador_e2.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {out_path}")

In [ ]:
# Figura 5 - Comparativa por modelo entre E1 (in-dist), E1b (cross-arch GAN) y E2 (cross-gen difusion)
# Se leen las metricas de E1/E1b desde los metrics.json de cada modelo en Drive.
modelo_folders = {
    'ResNet-50':           Path(modelos_info['ResNet-50']['checkpoint']).parent,
    'ViT-B/16':            Path(modelos_info['ViT-B/16']['checkpoint']).parent,
    'UniversalFakeDetect': Path(modelos_info['UniversalFakeDetect']['checkpoint']).parent,
}

auc_por_ronda = {}
for modelo, folder in modelo_folders.items():
    mpath = folder / 'metrics.json'
    with mpath.open() as f:
        m = json.load(f)
    auc_por_ronda[modelo] = {
        'E1 (ProGAN test)':         m['e1']['auc_roc'],
        'E1b (cross-arch GAN)':     m['e1b']['auc_roc'],
        'E2 (cross-gen difusion)':  metricas_global[modelo]['auc_roc'],
    }

rondas = ['E1 (ProGAN test)', 'E1b (cross-arch GAN)', 'E2 (cross-gen difusion)']
fig, ax = plt.subplots(figsize=(11, 6))
width = 0.25
x = np.arange(len(rondas))
offsets = [-width, 0.0, width]

for modelo, offset in zip(MODELOS, offsets):
    valores = [auc_por_ronda[modelo][r] for r in rondas]
    bars = ax.bar(x + offset, valores, width, label=modelo, color=COLORES[modelo], alpha=0.9, edgecolor='white')
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels(rondas, fontsize=11)
ax.set_ylim(0.0, 1.05)
ax.set_ylabel('AUC-ROC', fontsize=12)
ax.set_title('Degradado por dominio: GAN in-dist -> GAN cross-arch -> Difusion', fontsize=13)
ax.legend(fontsize=11, loc='lower left')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
out_path = OUT_DIR / 'comparativa_gan_vs_difusion.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {out_path}")

## 9. Resumen final

In [ ]:
print('=' * 60)
print('  Resumen E2 (GenImage cross-generator difusion)')
print('=' * 60)

print('\nAgregado (todo GenImage val):')
print(f"{'Modelo':<22s}  {'AUC':>8s}  {'AP':>8s}  {'Acc':>8s}  {'N':>7s}")
print('-' * 60)
for n, m in metricas_global.items():
    print(f"{n:<22s}  {m['auc_roc']:>8.4f}  {m['average_precision']:>8.4f}  {m['accuracy']:>8.4f}  {m['n_samples']:>7d}")

print('\nDegradado de AUC por ronda:')
print(f"{'Modelo':<22s}  {'E1':>7s}  {'E1b':>7s}  {'E2':>7s}  {'E1->E2':>8s}")
print('-' * 60)
for modelo in MODELOS:
    a1   = auc_por_ronda[modelo]['E1 (ProGAN test)']
    a1b  = auc_por_ronda[modelo]['E1b (cross-arch GAN)']
    a2   = auc_por_ronda[modelo]['E2 (cross-gen difusion)']
    drop = a1 - a2
    print(f"{modelo:<22s}  {a1:>7.4f}  {a1b:>7.4f}  {a2:>7.4f}  {drop:>+8.4f}")

print('\nFicheros generados en', OUT_DIR, ':')
for f in sorted(OUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45s}  {size_kb:>8.1f} KB")